# GGAndy Odoo 19 資料關係 Playground

這份 Notebook 用來**讀取目前 `ggandy_dev` 的真實資料**，幫你理解物件、出租單位、租約、房客與租金期次的關係。

建議操作方式：用 VS Code 連入 `ggandy_odoo19_web` 容器，開啟 `/mnt/notebook/2026-09-15_playground.ipynb`，Kernel 選擇容器內的 **Python 3 (ipykernel)**，再由上到下執行。

## 先看懂整體關係

```text
房東 res.partner
       ↑ owner_id（Many2one）
物件 ggandy.property
       ↓ unit_ids（One2many）
出租單位 ggandy.property.unit
       ↓ lease_ids（One2many）
租約 ggandy.lease ── tenant_id（Many2one）──→ 房客 res.partner
       ↓ schedule_ids（One2many）
租金期次 ggandy.rent.schedule ── invoice_id（Many2one）──→ Odoo 發票
```

一棟物件可以有很多出租單位；一個出租單位可以累積很多歷史租約；每張租約可以產生很多個月的租金期次。

## 1. 建立 Odoo Shell 環境

下面這格相當於替 Notebook 建立一個可使用的 Odoo `env`。它使用容器環境變數連線，不會把資料庫密碼寫進 Notebook。

In [ ]:
import os
from odoo import api, SUPERUSER_ID
from odoo.modules.registry import Registry
from odoo.tools import config

# 重跑此格時，先關閉上一條資料庫連線。
old_cursor = globals().get("cr")
if old_cursor:
    try:
        old_cursor.close()
    except Exception:
        pass

DB_NAME = "ggandy_dev"
config.parse_config([
    "--config=/etc/odoo/odoo.conf",
    f"--database={DB_NAME}",
    "--db_host=" + os.environ.get("HOST", "ggandy_db"),
    "--db_port=" + os.environ.get("PORT", "5432"),
    "--db_user=" + os.environ.get("POSTGRES_USER", "ggandy_odoo"),
    "--db_password=" + os.environ.get("POSTGRES_PASSWORD", ""),
    "--no-http",
])

registry = Registry(DB_NAME)
cr = registry.cursor()
env = api.Environment(cr, SUPERUSER_ID, {})

print(f"已連線：{DB_NAME}")
print(f"公司：{env.company.name}")

## 2. Odoo ORM 的三種常見關係

- **Many2one**：目前這筆資料只指向一筆，例如出租單位的 `property_id` 指向一棟物件。
- **One2many**：從另一側看回來的多筆資料，例如物件的 `unit_ids` 是它底下所有出租單位。
- **Many2many**：兩邊都可以有多筆，例如物件的 `co_owner_ids` 可以有多位共同屋主，同一人也可能是多個物件的共同屋主。

`One2many` 通常不是自己存一串 ID；真正存放關係的是另一個模型上的 `Many2one`。例如 `property.unit` 資料表存有 `property_id`。

In [ ]:
models_to_count = {
    "物件": "ggandy.property",
    "出租單位": "ggandy.property.unit",
    "租約": "ggandy.lease",
    "租金期次": "ggandy.rent.schedule",
}

for label, model_name in models_to_count.items():
    print(f"{label}：{env[model_name].search_count([])} 筆")

## 3. 從物件找到出租單位

`env[模型名稱].search([])` 代表搜尋全部資料。下面從每個 `property` 的 `unit_ids` 往下走。

In [ ]:
properties = env["ggandy.property"].search([], order="code, name")

for property_record in properties:
    print(f"\n🏠 {property_record.code}｜{property_record.name}")
    print(f"   房東：{property_record.owner_id.name}")
    print(f"   地址：{property_record.address_display}")
    print(f"   出租單位：{len(property_record.unit_ids)} 間")
    for unit in property_record.unit_ids:
        print(f"   └─ {unit.name}｜{unit.area:g} 坪｜月租 {unit.monthly_rent:,.0f}｜{unit.state}")

## 4. 從出租單位反查物件

剛才使用物件的 `unit_ids`（One2many）。現在反過來使用出租單位的 `property_id`（Many2one）。它們是同一段關係的兩個方向。

In [ ]:
units = env["ggandy.property.unit"].search([], order="property_id, floor, name")

for unit in units:
    print(f"{unit.display_name} → 所屬物件：{unit.property_id.name}｜歷史租約：{len(unit.lease_ids)} 張")

## 5. 租約把出租單位與房客接起來

租約的 `unit_id` 指向出租單位，`tenant_id` 指向主承租人。租約生效時，系統會把出租單位改成「已出租」，並依月份建立 `schedule_ids`。目前若尚未建立 Demo 租約，下面會顯示提示，而不是出錯。

In [ ]:
leases = env["ggandy.lease"].search([], order="start_date desc")

if not leases:
    print("目前沒有租約資料。之後在 Odoo 建立租約，再重跑這格就能看到關係。")

for lease in leases:
    print(f"\n📄 {lease.name}｜{lease.state}")
    print(f"   物件：{lease.property_id.name}")
    print(f"   單位：{lease.unit_id.name}")
    print(f"   房客：{lease.tenant_id.name}")
    print(f"   租期：{lease.start_date} ～ {lease.end_date}")
    print(f"   月租：{lease.rent_amount:,.0f}｜租金期次：{len(lease.schedule_ids)} 筆")

In [ ]:
schedules = env["ggandy.rent.schedule"].search([], order="due_date desc", limit=12)

if not schedules:
    print("目前沒有租金期次。租約生效後可以自動產生。")

for schedule in schedules:
    print(
        f"{schedule.name}｜{schedule.tenant_id.name}｜"
        f"應收 {schedule.total_amount:,.0f}｜到期 {schedule.due_date}｜{schedule.collection_state}"
    )

## 6. 兩個很好用的 Recordset 操作

- `filtered()`：從查到的紀錄中篩選，例如只留下空房。
- `mapped()`：沿欄位關係取值，例如從所有單位取出它們所屬的物件。

In [ ]:
vacant_units = units.filtered(lambda unit: unit.state == "vacant")
related_properties = units.mapped("property_id")

print("空房：", vacant_units.mapped("display_name"))
print("這些單位涉及的物件：", related_properties.mapped("name"))

## 7. 結束前關閉交易

這份 Playground 預設只做查詢。若日後練習 `create()` 或 `write()`：

- 想保留異動才執行 `cr.commit()`。
- 只是實驗就執行 `cr.rollback()`。

下面會 rollback 並關閉連線，確保練習不會意外改到資料。執行後若想再查詢，重新執行第 1 節的連線格即可。

In [ ]:
cr.rollback()
cr.close()
print("已 rollback 並關閉 Odoo 資料庫連線。")